# 문제해결: 금융사기 보고서 기반 FAISS Naive RAG

## 실습 목적

Naive RAG의 기본 흐름을 익힌 후, **직접 문서 기반 질의응답 시스템을 구성하고 문제를 점검**하도록 설계한 실습 예제

사용 문서: `data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf`

## 실습 시나리오

금융소비자 교육 담당자가 하나의 금융사기 보고서를 기반으로 질문에 답변하는 내부용 QA 챗봇을 만들려고 한다.  

## 완성 기준

| 기준 | 설명 |
|---|---|
| 문서 로딩 | PDF 페이지를 `Document` 객체로 정상 로딩한다. |
| 청킹 | 문서가 적절한 크기의 청크로 분할된다. |
| 벡터DB 생성 | FAISS 벡터스토어가 생성되고 로컬 저장까지 가능하다. |
| 검색 | 질문과 관련된 상위 문서 조각을 검색한다. |
| 답변 생성 | 검색된 문서에 근거해서만 답변한다. |
| 출처 확인 | 답변 근거가 되는 페이지와 청크를 확인할 수 있다. |
| 문제해결 | 검색 실패, 근거 부족, 청크 크기 문제를 점검할 수 있다. |

## 0. 설치

> `faiss-cpu`는 로컬 CPU 환경에서 FAISS를 사용하기 위한 패키지이다.

In [1]:
# uv add langchain langchain-openai langchain-community langchain-text-splitters pypdf faiss-cpu python-dotenv pandas

## 1. 라이브러리 불러오기

LangChain의 기본 구성요소를 사용한다.

- `PyPDFLoader`: PDF 문서 로딩
- `RecursiveCharacterTextSplitter`: 문서 청킹
- `OpenAIEmbeddings`: 텍스트 임베딩
- `FAISS`: 로컬 벡터스토어
- `ChatOpenAI`: 답변 생성 LLM

In [2]:
from pathlib import Path
import os
from pprint import pprint

import pandas as pd
from dotenv import load_dotenv

from langchain_pymupdf4llm import PyMuPDF4LLMLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

C:\Users\magpi\AppData\Local\Temp\ipykernel_29652\4097930351.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


## 2. 환경변수 설정

OpenAI API Key는 `.env` 파일 또는 운영체제 환경변수에 설정한다.

`.env` 파일 예시는 다음과 같다.

```text
OPENAI_API_KEY=sk-...
```

In [3]:
load_dotenv(override=True, dotenv_path="../.env")

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError(
        "OPENAI_API_KEY가 설정되어 있지 않습니다. .env 파일 또는 환경변수에 OPENAI_API_KEY를 설정하세요."
    )

print("OPENAI_API_KEY 설정 확인 완료")

OPENAI_API_KEY 설정 확인 완료


## 3. PDF 파일 경로 설정

노트북과 같은 폴더 또는 `data` 폴더에 PDF를 넣어두면 자동으로 찾는다.

권장 폴더 구조는 다음과 같다.

```text
project/
├─ 1_faiss_naive_rag_financial_fraud.ipynb
└─ data/
   └─ 진화하는 금융사기, 모두의 경계가 필요한 시점.pdf
```

In [4]:
PDF_NAME = "진화하는 금융사기, 모두의 경계가 필요한 시점.pdf"

candidate_paths = [
    Path(PDF_NAME),
    Path("data") / PDF_NAME,
    Path("/mnt/data") / PDF_NAME,  # ChatGPT 작업환경용 경로
]

PDF_PATH = next((path for path in candidate_paths if path.exists()), None)

if PDF_PATH is None:
    raise FileNotFoundError(
        f"PDF 파일을 찾을 수 없습니다. 다음 위치 중 하나에 파일을 넣어주세요: {candidate_paths}"
    )

print("PDF_PATH:", PDF_PATH)

PDF_PATH: data\진화하는 금융사기, 모두의 경계가 필요한 시점.pdf


## 4. PDF 로딩

PDF를 페이지 단위 `Document`로 로딩한다.  
각 페이지는 `page_content`와 `metadata`를 가진다.

### pdf 파일 로딩

In [22]:
loader = PyMuPDF4LLMLoader(str(PDF_PATH))
pages = loader.load()


from pathlib import Path
# Markdown 저장 경로 설정

OUTPUT_DIR = Path("./data")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MD_PATH = OUTPUT_DIR / f"{PDF_PATH.stem}_pymupdf4llm.md"
md_blocks = []

for doc in pages:
    page_label = doc.metadata.get("page_label", doc.metadata.get("page", 0) + 1)
    
    md_blocks.append(f"\n\n---\n\n# Page {page_label}\n\n")
    md_blocks.append(doc.page_content.strip())

full_markdown = "\n".join(md_blocks)

with open(MD_PATH, "w", encoding="utf-8") as f:
    f.write(full_markdown)

print("Markdown 저장 완료:")
print(MD_PATH.resolve())

Markdown 저장 완료:
E:\langgraph_modular_rag\실습문제\data\진화하는 금융사기, 모두의 경계가 필요한 시점_pymupdf4llm.md


In [ ]:
# 메타데이터 보강
for doc in pages:
    doc.metadata["source_name"] = PDF_NAME
    doc.metadata["page_label"] = doc.metadata.get("page", 0) + 1

print("로드된 페이지 수:", len(pages))
print("첫 번째 페이지 메타데이터:")
pprint(pages[0].metadata)
print("\n첫 번째 페이지 내용 일부:")
print(pages[0].page_content[:500])

로드된 페이지 수: 15
첫 번째 페이지 메타데이터:
{'author': 'user',
 'creationDate': "D:20260529090329+09'00'",
 'creationdate': '2026-05-29T09:03:29+09:00',
 'creator': 'Hancom PDF 1.3.0.534',
 'file_path': 'data\\진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'format': 'PDF 1.4',
 'keywords': '',
 'modDate': "D:20260529090329+09'00'",
 'moddate': '2026-05-29T09:03:29+09:00',
 'page': 0,
 'page_label': 1,
 'producer': 'Hancom PDF 1.3.0.534',
 'source': 'data\\진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'source_name': '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'subject': '',
 'title': '',
 'total_pages': 15,
 'trapped': ''}

첫 번째 페이지 내용 일부:
2026년 5월 29일 제 17 호

## 진화하는 금융사기, 모두의 경계가 필요한 시점





## 5. 문서 청킹

RAG에서는 문서 전체를 한 번에 넣지 않고, 검색 가능한 단위로 나눈다.  
이 실습에서는 보고서형 PDF에 적합한 기본값으로 `chunk_size=800`, `chunk_overlap=120`을 사용한다.

- `chunk_size`: 하나의 문서 조각 최대 길이
- `chunk_overlap`: 앞뒤 청크가 겹치는 길이
- 목적: 문맥 손실을 줄이면서 검색 단위를 적절히 유지하는 것

In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", " ", ""],
)

chunks = text_splitter.split_documents(pages)

for i, doc in enumerate(chunks):
    doc.metadata["chunk_id"] = i
    doc.metadata["page_label"] = doc.metadata.get("page", 0) + 1
    doc.metadata["source_name"] = PDF_NAME

print("전체 청크 수:", len(chunks))
print("첫 번째 청크 메타데이터:")
pprint(chunks[0].metadata)
print("\n첫 번째 청크 내용:")
print(chunks[0].page_content[:800])

전체 청크 수: 28
첫 번째 청크 메타데이터:
{'author': 'user',
 'chunk_id': 0,
 'creationDate': "D:20260529090329+09'00'",
 'creationdate': '2026-05-29T09:03:29+09:00',
 'creator': 'Hancom PDF 1.3.0.534',
 'file_path': 'data\\진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'format': 'PDF 1.4',
 'keywords': '',
 'modDate': "D:20260529090329+09'00'",
 'moddate': '2026-05-29T09:03:29+09:00',
 'page': 0,
 'page_label': 1,
 'producer': 'Hancom PDF 1.3.0.534',
 'source': 'data\\진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'source_name': '진화하는 금융사기, 모두의 경계가 필요한 시점.pdf',
 'subject': '',
 'title': '',
 'total_pages': 15,
 'trapped': ''}

첫 번째 청크 내용:
2026년 5월 29일 제 17 호

## 진화하는 금융사기, 모두의 경계가 필요한 시점


## 6. 청크 분포 확인

페이지별 청크 수를 확인한다.  
특정 페이지의 청크가 지나치게 많거나 적으면 청킹 전략을 조정해야 한다.

In [7]:
chunk_df = pd.DataFrame([
    {
        "chunk_id": doc.metadata["chunk_id"],
        "page": doc.metadata.get("page_label"),
        "text_length": len(doc.page_content),
        "preview": doc.page_content[:80].replace("\n", " ")
    }
    for doc in chunks
])

print("페이지별 청크 수")
display(chunk_df.groupby("page").size().reset_index(name="chunk_count"))

print("청크 샘플")
display(chunk_df.head(10))

페이지별 청크 수


,page,chunk_count
0,1,1
1,2,2
2,3,2
3,4,2
4,5,2
5,6,2
6,7,3
7,8,2
8,9,2
9,10,2


청크 샘플


,chunk_id,page,text_length,preview
0,0,1,49,"2026년 5월 29일 제 17 호 ## 진화하는 금융사기, 모두의 경계가 필요한 시점"
1,1,2,725,##### < Executive Summary > ###### ∎ 금융사기는 의도...
2,2,2,358,l 최근에는 AI로 목소리나 화면을 조작하여 피해자가 비대면으로 진위 여부를 확인할...
3,3,3,772,"###### ∎ 규제 공백 해소를 위한 입법 노력이 지속되고 있으며, 유관기관 간 ..."
4,4,3,234,"l 개별법 제·개정, 규제 강화 중심의 예방 노력은 후행적 조치라는 한계가 존재하므..."
5,5,4,784,### I. 금융사기란? ##### 1. 금융사기의 정의와 특징 ###### ■...
6,6,4,376,할 수 없도록 사기 행각을 벌이는 등 기술을 이용한 수법이 고도화 표1 | 금융...
7,7,5,781,##### 2. 금융사기 주요 유형 ###### ■ 금융사기는 금전적 피해를 유발...
8,8,5,609,"사기 행위를 분류하며, 수법에 따라 관련 규정을 지속적으로 확대하는 추세 표2 ..."
9,9,6,780,### II. 금융사기 피해 현황 ###### ■ ’24년 기준 최근 2년 내 금...


## 7. 임베딩 모델과 FAISS 벡터스토어 생성

여기서부터 인덱싱 단계이다.

```text
청크 텍스트 → 임베딩 벡터 → FAISS 인덱스 저장
```

FAISS는 벡터 간 유사도 검색을 빠르게 수행하기 위한 로컬 벡터 검색 라이브러리이다.  
이 노트북에서는 LangChain의 `FAISS.from_documents()`를 사용해 문서 청크와 임베딩 모델을 연결한다.

In [8]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings,
)

print("FAISS 벡터스토어 생성 완료")
print("저장된 문서 청크 수:", vectorstore.index.ntotal)

FAISS 벡터스토어 생성 완료
저장된 문서 청크 수: 28


## 8. 기본 유사도 검색 테스트

먼저 LLM 답변 생성 없이, 검색 결과만 확인한다.  
이 단계가 중요하다. 검색이 잘못되면 답변도 잘못될 가능성이 높다.

`similarity_search_with_score()`의 score는 기본 FAISS 설정에서는 거리 기반 점수로 해석한다.  
일반적으로 **낮을수록 더 가까운 문서**로 볼 수 있다.

In [9]:
question = "금융사기는 어떤 단계로 발생하나요?"

results = vectorstore.similarity_search_with_score(question, k=4)

for rank, (doc, score) in enumerate(results, start=1):
    print(f"\n[{rank}] score={score:.4f} | page={doc.metadata.get('page_label')} | chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:500].replace("\n", " "))


[1] score=0.8150 | page=4 | chunk=5
### I. 금융사기란?  ##### 1. 금융사기의 정의와 특징  ###### ■ 금융사기는 사람을 속이거나 착각하게 하여 금전적 이득을 취하는 범죄 행위      - 금융사기는 사기 수법과 수단에 따라 양상이 다르게 나타나나, 공통적으로 ‘신뢰  유도-개인정보 수집-금전 편취’ 단계에 걸쳐 금전적 피해를 유발      택배회사 발송 배송조회 QR코드(신뢰)로 위장하여 피해자가 코드 스캔 시 악성코드나   앱을 스마트폰에 설치(수집)하고, 이를 통해 탈취한 개인정보로 금전을 유출(탈취)      유명 핀플루언서(금융+인플루언서)의 영상을 도용(신뢰)하여 가짜 채널로 가입을 권유   (수집)한 후 미끼 수익을 제공(신뢰)하고 더 큰 금액의 투자를 유도(탈취)한 뒤 잠적  ###### ■ 대면에 기반한 과거 사기 수법과는 달리 기술의 발달로 익명성·비대면성이 보편화됨에 따라 불특정 다수를 대상으로 대규모 피해를 유발하는 범죄 확대      - 해외에 사무소를 두고 조직적으로 범죄에 가

[2] score=0.8297 | page=3 | chunk=4
l 개별법 제·개정, 규제 강화 중심의 예방 노력은 후행적 조치라는 한계가 존재하므로   금융사기 예방을 위한 범기관 네트워크 구축이 요구   l 금융사는 금융사기 전담 부서 조직을 확충하고 금융사기 예방 시스템을 운영하는   한편, 피해 발생 시 배상 프로세스 기준과 절차를 선제적으로 마련할 필요   연 구 원 서 가 연 gayeonseo@hanafn.com   Key Words : 금융사기, 보이스피싱, 투자사기

[3] score=0.8707 | page=13 | chunk=25
- 금융거래 데이터뿐만 아니라 앱 이용 패턴 데이터, 통신 데이터 등 사기 거래 전반에   필요한 데이터를 종합적으로 분석하고 학습하여 사기를 예방할 수 있는 시스템 구축       - 금융회사가 자체적으로 개발한 금융사기 탐지 프로그램을 타 금융사와 공유하는 등  사회

## 9. Retriever 구성

Retriever는 사용자의 질문을 받아 관련 문서 조각을 반환하는 객체이다.

```text
질문 → Retriever → 관련 청크 Top-k 반환
```

In [10]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

retrieved_docs = retriever.invoke("전기통신금융사기와 투자사기의 차이는 무엇인가요?")

for doc in retrieved_docs:
    print(f"page={doc.metadata.get('page_label')}, chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:300].replace("\n", " "))
    print("-" * 80)

page=5, chunk=7
##### 2. 금융사기 주요 유형  ###### ■ 금융사기는 금전적 피해를 유발하는 범죄행위 전반을 포괄해 다양한 기준으로 구분되나 전기통신금융사기와 투자사기 두 분류로 구분하는 것이 보편적      - 전기통신금융사기는 전화, 메신저, 인터넷 등 전기통신을 이용하여 금융기관·   지인·가족 등을 사칭하거나 허위 사실로 피해자의 자금을 탈취하는 수법      피싱, 스미싱, 파밍 등이 대표적인 유형이며, 법에서 열거하는 통신수단을 이용하여   개인정보를 노출시켜 자금을 송금·이체하도록 유도      ’11년 시행된 통신사기피해환
--------------------------------------------------------------------------------
page=7, chunk=12
여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹 안심차단(’25.11)을   차례로 시행하여 금융소비자가 자발적 신청을 통해 금융사기를 예방하도록 장려      - 금융소비자가 직접 신청해야 하는 서비스 외에 자금 이체 시 수취인 계좌에 일정 시간  경과 후 입금되는 지연이체 서비스 등 부정출금 방지를 위한 거래 방식을 도입   표3 | 전기통신금융사기 주요 대응 현황 표4 | 투자사기 주요 대응 현황                ’24.8 여신거래 안심차단 서비스 시행 가상자산이용자 Ÿ 미공개정보이용
--------------------------------------------------------------------------------
page=5, chunk=8
사기 행위를 분류하며, 수법에 따라 관련 규정을 지속적으로 확대하는 추세   표2 | 금융사기 분류 그림2 | 금융사기 접근 경로         |분류|하위종류|Col3| |---|---|---| |전기통신<br>금융사기|보이스피싱|전화로 타인을 사칭하여 송금을 유도하는 수법| |전기통신<br>금융사기|메신저피싱|카카오톡이나 SNS로

## 10. RAG 답변 프롬프트 구성

핵심은 LLM이 검색된 문서 밖의 내용을 임의로 생성하지 않도록 제약하는 것이다.

프롬프트 설계 원칙은 다음과 같다.

1. 문서 근거만 사용한다.
2. 문서에서 확인할 수 없으면 모른다고 답한다.
3. 답변 마지막에 근거 페이지를 포함한다.

In [11]:
def format_docs(docs):
    """검색된 문서들을 LLM 입력용 context 문자열로 변환한다."""
    formatted = []
    for doc in docs:
        page = doc.metadata.get("page_label", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        formatted.append(
            f"[출처: page {page}, chunk {chunk_id}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
당신은 금융사기 보고서 기반 QA 어시스턴트입니다.
아래 [문서]에 근거해서만 답변하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다.'라고 답변하세요.
답변은 핵심 위주로 3~5문장으로 정리하세요.
답변 마지막에 근거 페이지를 '근거: p.숫자' 형식으로 표시하세요.

[문서]
{context}
""".strip()
    ),
    ("human", "{question}"),
])

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

answer_chain = prompt | llm | StrOutputParser()

## 11. RAG 함수 만들기

검색과 답변 생성을 하나의 함수로 묶는다.

```text
질문 입력
→ FAISS 검색
→ 검색 문서 format
→ LLM 답변 생성
→ 답변과 출처 반환
```

In [12]:
def answer_question(question: str, k: int = 4):
    """질문에 대해 FAISS 검색 후 문서 기반 답변과 출처 테이블을 반환한다."""
    docs = vectorstore.similarity_search(question, k=k)
    context = format_docs(docs)

    answer = answer_chain.invoke({
        "question": question,
        "context": context,
    })

    sources = pd.DataFrame([
        {
            "rank": i + 1,
            "page": doc.metadata.get("page_label"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "preview": doc.page_content[:120].replace("\n", " ")
        }
        for i, doc in enumerate(docs)
    ])

    return answer, sources

## 12. RAG 실행 예시

다음 질문들은 이 보고서에 포함된 내용을 기반으로 답변 가능한 질문이다.

In [13]:
question = "전기통신금융사기와 투자사기는 어떻게 다른가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
전기통신금융사기와 투자사기는 어떻게 다른가요?

[답변]
전기통신금융사기는 전화, 메신저, 인터넷 등을 이용해 금융기관이나 지인 등을 사칭하여 피해자의 자금을 탈취하는 범죄로, 피싱, 스미싱, 파밍 등이 포함됩니다. 반면, 투자사기는 투자자의 수익 창출 욕구를 이용해 투자를 유도하거나 가격을 조작하여 금전적 피해를 유발하는 범죄로, 허위 수익률을 제시하거나 가짜 투자 플랫폼을 이용하는 방식이 있습니다. 두 유형은 범죄 수법과 피해자의 접근 방식에서 차이를 보입니다. 근거: p.5

[검색된 근거]


,rank,page,chunk_id,preview
0,1,5,7,##### 2. 금융사기 주요 유형 ###### ■ 금융사기는 금전적 피해를 유발...
1,2,7,12,"여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹 안..."
2,3,3,4,"l 개별법 제·개정, 규제 강화 중심의 예방 노력은 후행적 조치라는 한계가 존재하므..."
3,4,5,8,"사기 행위를 분류하며, 수법에 따라 관련 규정을 지속적으로 확대하는 추세 표2 ..."


In [14]:
question = "금융회사들은 금융사기를 예방하기 위해 어떤 시스템을 도입하고 있나요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
금융회사들은 금융사기를 예방하기 위해 어떤 시스템을 도입하고 있나요?

[답변]
금융회사들은 비정상적 금융거래를 탐지하고 차단하기 위해 이상금융거래탐지시스템(Fraud Detection System)을 도입하고 있습니다. 이 시스템은 거래 내역, 고객 정보, 평소 거래 패턴 등을 분석하여 이상 거래를 차단합니다. 또한, AI 기술을 활용한 탐지 시스템 고도화와 관련 기관과의 협업을 통해 금융사기 예방을 위한 체계를 강화하고 있습니다. 근거: p.9, p.12

[검색된 근거]


,rank,page,chunk_id,preview
0,1,9,16,##### 2. 금융회사 책임 강화 ###### ■ 금융회사는 비정상적 금융거래로...
1,2,13,25,"- 금융거래 데이터뿐만 아니라 앱 이용 패턴 데이터, 통신 데이터 등 사기 거래 전..."
2,3,12,23,###### ■ 금융사기 수법 고도화에 따라 금융회사도 각종 탐지 및 예방 기술을 ...
3,4,12,22,###### ■ 금융사는 자체적으로 금융사기를 예방하기 위한 장치를 마련하고 유관 ...


## 13. 문서에 없는 질문 테스트

RAG 실습에서는 답변 가능한 질문만 넣으면 안 된다.  
문서에 없는 질문을 넣어 **모른다고 답하는지** 확인해야 한다.

In [15]:
question = "2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?

[답변]
문서에서 확인할 수 없습니다.

[검색된 근거]


,rank,page,chunk_id,preview
0,1,10,19,보상한도를 5천만 원으로 설정할 경우 ’24년 보이스피싱 피해자의 약 85.2%가 ...
1,2,6,10,- 투자사기의 ’25년 월평균 피해액은 548억 원으로 ’23년 317억 원과 비교...
2,3,6,9,### II. 금융사기 피해 현황 ###### ■ ’24년 기준 최근 2년 내 금...
3,4,10,18,###### ■ 또한 사전적 예방 의무에 그치지 않고 보이스피싱 사고로 발생한 금전...


## 14. FAISS 인덱스 저장과 재로딩

실제 서비스에서는 매번 PDF를 읽고 임베딩하면 비효율적이다.  
한 번 만든 FAISS 인덱스를 로컬에 저장한 뒤 다시 불러올 수 있다.

주의: `allow_dangerous_deserialization=True`는 신뢰할 수 있는 로컬 파일을 불러올 때만 사용한다.

In [16]:
INDEX_DIR = Path("faiss_financial_fraud_index")

vectorstore.save_local(str(INDEX_DIR))
print("FAISS 인덱스 저장 완료:", INDEX_DIR.resolve())

loaded_vectorstore = FAISS.load_local(
    folder_path=str(INDEX_DIR),
    embeddings=embeddings,
    allow_dangerous_deserialization=True,
)

print("FAISS 인덱스 재로딩 완료")
print("재로딩된 문서 청크 수:", loaded_vectorstore.index.ntotal)

FAISS 인덱스 저장 완료: E:\langgraph_modular_rag\실습문제\faiss_financial_fraud_index
FAISS 인덱스 재로딩 완료
재로딩된 문서 청크 수: 28


## 15. 간단 검색 평가

정교한 RAG 평가는 별도 설계가 필요하지만, 수업에서는 우선 **Context Hit** 방식으로 검색 결과를 점검할 수 있다.

여기서는 기대 키워드가 검색된 상위 문서 안에 포함되어 있는지 확인한다.

In [17]:
eval_questions = [
    {
        "question": "금융사기는 일반적으로 어떤 두 가지로 구분되나요?",
        "expected_keywords": ["전기통신금융사기", "투자사기"],
    },
    {
        "question": "ASAP 플랫폼은 어떤 정보를 공유하나요?",
        "expected_keywords": ["보이스피싱", "정보공유", "AI플랫폼"],
    },
    {
        "question": "은행권 자율배상 제도는 무엇인가요?",
        "expected_keywords": ["자율배상", "보이스피싱", "개인정보"],
    },
    {
        "question": "해외에서는 금융사기에 어떻게 대응하고 있나요?",
        "expected_keywords": ["호주", "영국", "일본"],
    },
]

rows = []

for item in eval_questions:
    docs = vectorstore.similarity_search(item["question"], k=4)
    context = "\n".join(doc.page_content for doc in docs)
    hit_keywords = [kw for kw in item["expected_keywords"] if kw in context]
    rows.append({
        "question": item["question"],
        "expected_keywords": ", ".join(item["expected_keywords"]),
        "hit_keywords": ", ".join(hit_keywords),
        "hit_count": len(hit_keywords),
        "top_pages": sorted({doc.metadata.get("page_label") for doc in docs}),
    })

eval_df = pd.DataFrame(rows)
display(eval_df)

,question,expected_keywords,hit_keywords,hit_count,top_pages
0,금융사기는 일반적으로 어떤 두 가지로 구분되나요?,"전기통신금융사기, 투자사기","전기통신금융사기, 투자사기",2,"[3, 4, 5]"
1,ASAP 플랫폼은 어떤 정보를 공유하나요?,"보이스피싱, 정보공유, AI플랫폼","보이스피싱, 정보공유, AI플랫폼",3,"[3, 8, 14]"
2,은행권 자율배상 제도는 무엇인가요?,"자율배상, 보이스피싱, 개인정보","자율배상, 보이스피싱, 개인정보",3,"[10, 11]"
3,해외에서는 금융사기에 어떻게 대응하고 있나요?,"호주, 영국, 일본","호주, 영국, 일본",3,"[3, 11, 12, 13]"


## 17. 정리

이 실습에서 구현한 Naive RAG 흐름은 다음과 같다.

```text
PDF 로딩
→ 페이지 단위 Document 생성
→ RecursiveCharacterTextSplitter로 청킹
→ OpenAIEmbeddings로 임베딩
→ FAISS 벡터스토어 생성
→ 질문 기반 Top-k 검색
→ 검색 문서 기반 LLM 답변 생성
→ 출처 페이지 확인
```

## 핵심 포인

| 관찰 포인트 | 핵심 내용 |
|---|---|
| 검색 결과가 부정확함 | 청크 크기, 질문 표현, Top-k 조정 필요 |
| 답변이 문서 밖 내용을 포함함 | 프롬프트 제약 및 문서 없음 질문 테스트 필요 |
| 표·그림 내용이 누락됨 | PDF 파싱 방식의 한계, Markdown 변환 비교 필요 |
| 같은 질문인데 답변이 달라짐 | 검색 결과와 LLM 생성 조건을 분리해서 점검해야 함 |
| 인덱스 재사용 필요 | FAISS 저장·로드 구조로 인덱싱 비용 절감 가능 |